# 05A — Prepare BraTS 2D Dataset

**Run once. Save the output as a Kaggle dataset.**

### Datasets to attach before running
| Dataset | Where to find it | Role |
|---|---|---|
| `brats20-dataset-training-validation` by awsaf49 | Kaggle Datasets | Raw NIfTI MRI volumes |
| Your code zip (e.g. `brats-seg-code`) | Upload → Datasets → New | `src/`, `scripts/`, `configs/` |

### What this notebook does
1. Copies the code repo into `/kaggle/working/`
2. Converts raw 3D NIfTI volumes → 2D axial `.npy` slices (float16, tumor-only)
3. Creates a patient-level train/val/test split
4. Reports disk usage

### After it finishes
Click **Save Version** → set output to **Dataset** → name it `brats2020-2d-prepared`. 
Notebooks 05B / 05C / 05D will read from that dataset.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────
# Slug of your uploaded code dataset (the folder name under /kaggle/input/)
CODE_DATASET_SLUG = 'brats-seg-code'          # change if you named it differently

KEEP_EMPTY_RATIO   = 0.0      # 0.0 = tumor-only slices. Critical for disk budget.
IMAGE_DTYPE        = 'float16' # float16 halves disk vs float32. Do not change.
MIN_TUMOR_PIXELS   = 20       # drop micro-label noise
SEED               = 42

In [ ]:
import os, shutil, subprocess, sys, json
from pathlib import Path

def disk_gb(path='/'):
    st = os.statvfs(path)
    return round(st.f_bavail * st.f_frsize / 1e9, 2)

def folder_gb(path):
    return round(sum(p.stat().st_size for p in Path(path).rglob('*') if p.is_file()) / 1e9, 2)

print(f'Free disk at start: {disk_gb()} GB  (limit ~19.5 GB)')

In [ ]:
# ── 1. Find and copy code repo ─────────────────────────────────────────────
def looks_like_repo(p):
    return (p / 'src').exists() and (p / 'scripts').exists() and (p / 'configs').exists()

repo_input = Path('/kaggle/input') / CODE_DATASET_SLUG
if not looks_like_repo(repo_input):
    for p in Path('/kaggle/input').rglob('requirements.txt'):
        if looks_like_repo(p.parent):
            repo_input = p.parent
            break

if not looks_like_repo(repo_input):
    raise FileNotFoundError(
        f'Cannot find repo under /kaggle/input. '
        f'Upload your code zip as a dataset and set CODE_DATASET_SLUG correctly. '
        f'Checked: {repo_input}'
    )

PROJECT = Path('/kaggle/working/project')
if PROJECT.exists():
    shutil.rmtree(PROJECT)
shutil.copytree(
    repo_input, PROJECT,
    ignore=shutil.ignore_patterns('.git', '__pycache__', '.pytest_cache', 'data', 'outputs', 'checkpoints')
)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

# run() pins every child process to the project root so that
# `python -m scripts.foo` can resolve `src.*` imports via sys.path[0]=''.
def run(cmd):
    subprocess.check_call(cmd, cwd=str(PROJECT))

print('Repo:', repo_input, '→', PROJECT)
print('cwd:', Path.cwd())

In [ ]:
# ── 2. Install only what Kaggle doesn't already have ──────────────────────
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'nibabel', 'albumentations'])
print('Packages ready.')

In [ ]:
# ── 3. Find raw BraTS data ────────────────────────────────────────────────
raw_candidates = [
    Path('/kaggle/input/brats20-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'),
    Path('/kaggle/input/brats2020-dataset-training-validation/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'),
]
RAW_BRATS = next((p for p in raw_candidates if p.exists()), None)
if RAW_BRATS is None:
    hits = list(Path('/kaggle/input').rglob('MICCAI_BraTS2020_TrainingData'))
    RAW_BRATS = hits[0] if hits else None
if RAW_BRATS is None:
    raise FileNotFoundError('Cannot find MICCAI_BraTS2020_TrainingData. Attach the brats20-dataset-training-validation dataset.')

patients = sorted(p for p in RAW_BRATS.iterdir() if p.is_dir())
print(f'Raw BraTS: {RAW_BRATS}')
print(f'Patients:  {len(patients)}')

In [ ]:
# ── 4. Prepare 2D slices ──────────────────────────────────────────────────
DATA_2D    = Path('/kaggle/working/brats2020_2d')
SPLIT_FILE = PROJECT / 'configs/splits/default.json'

if not (DATA_2D / 'metadata.json').exists():
    print(f'Preparing slices (float16, tumor-only)  — expected ~15 min …')
    run([
        sys.executable, '-m', 'scripts.prepare_data',
        '--input',            str(RAW_BRATS),
        '--output',           str(DATA_2D),
        '--keep-empty-ratio', str(KEEP_EMPTY_RATIO),
        '--image-dtype',      IMAGE_DTYPE,
        '--min-tumor-pixels', str(MIN_TUMOR_PIXELS),
        '--seed',             str(SEED),
    ])
else:
    print('Slices already prepared:', DATA_2D)

with open(DATA_2D / 'metadata.json') as f:
    meta = json.load(f)
print(f'Slices: {meta["num_slices"]:,}  |  Patients: {meta["num_patients"]}')
print(f'Prepared data size: {folder_gb(DATA_2D)} GB')
print(f'Free disk remaining: {disk_gb()} GB')

In [ ]:
# ── 5. Patient split ──────────────────────────────────────────────────────
SPLIT_FILE.parent.mkdir(parents=True, exist_ok=True)
if not SPLIT_FILE.exists():
    run([
        sys.executable, '-m', 'scripts.make_splits',
        '--data-root', str(DATA_2D),
        '--output',    str(SPLIT_FILE),
        '--seed',      str(SEED),
    ])

with open(SPLIT_FILE) as f:
    split = json.load(f)
for name, ids in split.items():
    print(f'{name}: {len(ids)} patients')

In [ ]:
# ── 6. Assemble output for saving as a Kaggle dataset ─────────────────────
# Layout that downstream notebooks expect:
#   /kaggle/input/brats2020-2d-prepared/
#       data/brats2020_2d/   ← prepared slices + metadata.json
#       configs/splits/default.json

EXPORT = Path('/kaggle/working/brats2020_2d_prepared')
if EXPORT.exists():
    shutil.rmtree(EXPORT)
(EXPORT / 'configs/splits').mkdir(parents=True)

# Move (not copy) — avoids doubling disk usage
shutil.move(str(DATA_2D), str(EXPORT / 'data/brats2020_2d'))
shutil.copy2(SPLIT_FILE, EXPORT / 'configs/splits/default.json')

# Free repo copy — no longer needed in this session
shutil.rmtree(PROJECT, ignore_errors=True)

output_gb = folder_gb(EXPORT)
print(f'Output folder: {EXPORT}')
print(f'Output size:   {output_gb} GB')
print(f'Free disk:     {disk_gb()} GB')
print()
print('NEXT STEP: Save Version → Output → Dataset → name it  brats2020-2d-prepared')